# Aviothic Breast Cancer Detection - Expanded Training Pipeline

This notebook demonstrates an expanded training pipeline with data augmentation, K-Fold cross-validation, and ROC analysis.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import KFold
import pandas as pd
import seaborn as sns
import os
import random

In [ ]:
# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

In [ ]:
# Data augmentation transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Model definition using transfer learning
def create_model(num_classes=3, pretrained=True):
    model = models.resnet50(pretrained=pretrained)
    
    # Freeze early layers for fine-tuning
    for param in model.parameters():
        param.requires_grad = False
        
    # Replace classifier
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    
    return model

In [ ]:
# K-Fold Cross Validation setup
def kfold_cross_validation(dataset, k_folds=5):
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    return kfold.split(dataset)

In [ ]:
# Training function
def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
# Validation function
def validate_model(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Store probabilities for ROC analysis
            probs = torch.nn.functional.softmax(outputs, dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc, all_labels, all_probs

In [ ]:
# ROC Analysis function
def plot_roc_curves(all_labels, all_probs, num_classes=3):
    plt.figure(figsize=(10, 8))
    
    # For each class
    for i in range(num_classes):
        # Convert to binary classification problem
        binary_labels = [1 if label == i else 0 for label in all_labels]
        class_probs = [prob[i] for prob in all_probs]
        
        # Calculate ROC curve
        fpr, tpr, _ = roc_curve(binary_labels, class_probs)
        roc_auc = auc(fpr, tpr)
        
        # Plot ROC curve
        plt.plot(fpr, tpr, lw=2, 
                label=f'Class {i} (AUC = {roc_auc:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Multi-class Classification')
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
# Main training function with K-Fold CV
def train_with_kfold_cv(dataset, k_folds=5, num_epochs=10, batch_size=32, learning_rate=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # K-Fold splits
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    fold_results = []
    
    for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
        print(f"Fold {fold+1}/{k_folds}")
        
        # Sample elements randomly from dataset
        train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
        val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids)
        
        # Define data loaders
        train_loader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
        val_loader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)
        
        # Initialize model
        model = create_model(num_classes=3, pretrained=True)
        model = model.to(device)
        
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)
        
        # Training loop
        for epoch in range(num_epochs):
            train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc, _, _ = validate_model(model, val_loader, criterion, device)
            
            print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        # Final validation for this fold
        _, val_acc, all_labels, all_probs = validate_model(model, val_loader, criterion, device)
        fold_results.append({
            'fold': fold+1,
            'accuracy': val_acc,
            'labels': all_labels,
            'probs': all_probs
        })
        
        print(f"Fold {fold+1} completed with accuracy: {val_acc:.2f}%\n")
    
    return fold_results

In [ ]:
# Run the training (commented out for Colab)
# Uncomment and adjust paths for actual training

# # Load dataset
# dataset = datasets.ImageFolder(root='path/to/your/dataset', transform=train_transform)

# # Run K-Fold CV training
# results = train_with_kfold_cv(dataset, k_folds=5, num_epochs=10, batch_size=32, learning_rate=0.001)

# # Print average results
# avg_acc = np.mean([r['accuracy'] for r in results])
# print(f"Average accuracy across folds: {avg_acc:.2f}%")

# # Plot ROC curves for the last fold
# last_fold = results[-1]
# plot_roc_curves(last_fold['labels'], last_fold['probs'])

In [ ]:
# Save the final model (example)
# torch.save(model.state_dict(), 'breast_cancer_model.pth')